# CF and UGRID Standards
## xarray-cf and xugrid packages

**Duration: ~30 minutes**

This notebook uses real SHYFEM ocean model output for **Antsiranana Bay, Madagascar** to demonstrate two important metadata conventions and the Python packages that leverage them:

| Convention | Scope | Python Package |
|---|---|---|
| [CF Conventions](https://cfconventions.org/) | Variable semantics (names, units, axes) | [`cf_xarray`](https://cf-xarray.readthedocs.io/) |
| [UGRID Conventions](https://ugrid-conventions.github.io/ugrid-conventions/) | Unstructured mesh topology | [`xugrid`](https://deltares.github.io/xugrid/) |

The SHYFEM model runs on an unstructured triangular mesh — a good real-world example of why both conventions matter.

In [ ]:
import os
import numpy as np
import xarray as xr
import cf_xarray
import hvplot.xarray
import holoviews as hv
from dotenv import load_dotenv

hv.extension('bokeh')

_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env')

endpoint = os.environ['ENDPOINT_URL']
so = dict(anon=False, endpoint_url=endpoint)

---
## Part 1: CF Conventions and cf_xarray (≈15 min)

### 1.1 What are CF Conventions?

The [Climate and Forecast (CF) Metadata Conventions](https://cfconventions.org/) standardize how variables, coordinates, and units are described in NetCDF and Zarr files. Without them, you'd need to know that one model uses `lon_rho`, another uses `nav_lon`, another uses `x` — all for the same thing.

Key attributes:

| Attribute | Purpose | Example |
|---|---|---|
| `standard_name` | Canonical identity | `"sea_water_temperature"` |
| `units` | UDUNITS-compatible | `"degC"`, `"m s-1"`, `"days since 1970-01-01"` |
| `axis` | Coordinate type | `"X"`, `"Y"`, `"Z"`, `"T"` |
| `positive` | Vertical direction | `"up"` or `"down"` |
| `coordinates` | Auxiliary coordinates | `"lon lat depth"` |
| `cell_methods` | Statistical operation | `"time: mean"` |

The global `Conventions` attribute declares compliance, e.g. `"CF-1.8"`.

### 1.2 Open the Antsiranana SHYFEM dataset

Two NetCDF files on S3-compatible object storage (EGI/rustfs):
- **surf.nos.nc4** — salinity, temperature (node × level × time)
- **surf.ous.nc4** — water_level, u_velocity, v_velocity (node × time)

In [ ]:
base = 's3://protocoast-data/full_dataset/shyfem/antsiranana'

ds_nos = xr.open_dataset(f'{base}/surf.nos.nc4', engine='h5netcdf', storage_options=so)
ds_ous = xr.open_dataset(f'{base}/surf.ous.nc4', engine='h5netcdf', storage_options=so)

# Merge into a single dataset
ds = xr.merge([ds_nos, ds_ous], compat='override')
ds

### 1.3 Inspect the CF metadata

In [ ]:
# Global attributes
print("Global attributes:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

In [ ]:
# Variable-level CF attributes
for vname in list(ds.coords) + list(ds.data_vars):
    attrs = ds[vname].attrs
    sn = attrs.get('standard_name', '')
    units = attrs.get('units', '')
    axis = attrs.get('axis', '')
    if sn or axis:
        print(f"{vname:20s}  standard_name={sn!r:45s}  units={units!r:12s}  axis={axis!r}")

### 1.4 The `.cf` accessor from cf_xarray

Importing `cf_xarray` attaches a `.cf` accessor to every xarray object.  
It reads the CF attributes and lets you access coordinates **by role** rather than by name.

In [ ]:
# Summary of what cf_xarray detects
ds.cf.describe()

In [ ]:
# Access coordinates by CF role — works regardless of the variable name
lon = ds.cf['longitude']
lat = ds.cf['latitude']
t   = ds.cf['time']
z   = ds.cf['vertical']

print(f"longitude → '{lon.name}'")
print(f"latitude  → '{lat.name}'")
print(f"time      → '{t.name}'")
print(f"vertical  → '{z.name}'")

### 1.5 Access data variables by `standard_name`

In [ ]:
# Access variables by standard_name — model-agnostic
sst = ds.cf['sea_water_temperature']
sal = ds.cf['sea_water_salinity']
wl  = ds.cf['water_surface_height_above_reference_datum']

print(f"temperature → '{sst.name}', dims={sst.dims}")
print(f"salinity    → '{sal.name}', dims={sal.dims}")
print(f"water_level → '{wl.name}',  dims={wl.dims}")

### 1.6 Selection with `cf.sel()` and `cf.isel()`

Use CF axis keys (`T`, `Z`, `X`, `Y`) in place of coordinate variable names.

In [ ]:
# Select surface (first level) at last time step
# Using CF axis names — no need to know the variable is called 'level' or 'time'
temp_surf = ds['temperature'].cf.isel(Z=0, T=-1).load()
print(f"shape: {temp_surf.shape}, time: {temp_surf.cf['time'].values}")

In [ ]:
# Select a specific depth using CF axis Z — value in meters
temp_10m = ds['temperature'].cf.sel(Z=10, method='nearest').cf.isel(T=0).load()
print(f"Nearest depth level: {temp_10m.cf['vertical'].values:.0f} m")

### 1.7 Quick map of surface temperature

Even without a structured grid, we can scatter-plot node values by their lon/lat coordinates.

In [ ]:
import pandas as pd
import holoviews as hv
import geoviews as gv
import holoviews.operation.datashader as hd

hd.datashade.precompute = True

# Build a DataFrame of lon, lat, temperature at surface/last time
temp_surf = ds['temperature'].cf.isel(Z=0, T=-1).load()

df = pd.DataFrame({
    'lon': ds.cf['longitude'].values,
    'lat': ds.cf['latitude'].values,
    'temperature': temp_surf.values
})

points = gv.operation.project_points(gv.Points(df, kdims=['lon','lat'], vdims=['temperature']))
tiles  = gv.tile_sources.OSM

tiles * hd.rasterize(points).opts(
    cmap='plasma', colorbar=True, width=600, height=450,
    title='Antsiranana Bay — Surface Temperature (°C)'
)

### 1.8 Time series at a point using CF selection

In [ ]:
# Find nearest node to a specific lat/lon
import xoak

target_lon, target_lat = 49.29, -12.30   # Antsiranana Bay

ds.xoak.set_index([lon.name, lat.name], 'scipy_kdtree')
ds_point = xr.Dataset({'lon': ('pt', [target_lon]), 'lat': ('pt', [target_lat])})

# Load full time series at the nearest node, surface level
ts = ds['temperature'].cf.isel(Z=0).xoak.sel(
    **{lon.name: ds_point.lon, lat.name: ds_point.lat}
).load()

ts.hvplot(x=t.name, grid=True, title='Surface Temperature at Antsiranana Bay', ylabel='°C')

---
## Part 2: UGRID Conventions and xugrid (≈15 min)

### 2.1 What are UGRID Conventions?

[UGRID](https://ugrid-conventions.github.io/ugrid-conventions/) extends CF to describe **unstructured meshes** — grids where cells are not arranged in a regular lat/lon matrix.

**Why unstructured grids?**

| Structured Grid | Unstructured Grid |
|---|---|
| Regular lat/lon or curvilinear | Triangles, quadrilaterals, polygons |
| Easy to index `[i, j]` | Requires topology connectivity |
| Wastes cells on land | Concentrates resolution where needed |
| ROMS, MOM, NEMO | FVCOM, SCHISM, **SHYFEM**, Delft3D FM |

**UGRID topology elements:**
```
node  — the points (vertices) of the mesh
edge  — line segments connecting two nodes  
face  — polygonal cells (triangles, quads, ...)
```

A **mesh topology variable** (scalar with `cf_role: mesh_topology`) declares the mesh and references the connectivity arrays.

### 2.2 Does the SHYFEM dataset follow UGRID?

The Antsiranana dataset has CF-1.4 attributes and a `topology` variable, but let's check if it's fully UGRID-compliant:

In [ ]:
print("'topology' variable attributes:")
for k, v in ds['topology'].attrs.items():
    print(f"  {k}: {v}")

print("\n'element_index' variable attributes:")
for k, v in ds['element_index'].attrs.items():
    print(f"  {k}: {v}")

**The topology variable is missing the UGRID-required attributes:**
- `cf_role: mesh_topology` — tells tools this is a UGRID mesh variable
- `face_node_connectivity: element_index` — points to the connectivity array
- `node_coordinates: longitude latitude` — points to the node coordinate arrays

This is common for older CF-1.4 files — the mesh information is present but not yet labeled as UGRID. We can add these attributes to make the file xugrid-compatible.

### 2.3 Add UGRID attributes and open with xugrid

In [ ]:
import xugrid as xu

# Annotate the dataset with UGRID-required attributes
ds_ugrid = ds.copy()

# 1. The topology/mesh variable needs cf_role and pointers to coordinates/connectivity
ds_ugrid['topology'].attrs.update({
    'cf_role': 'mesh_topology',
    'topology_dimension': 2,
    'node_coordinates': 'longitude latitude',
    'face_node_connectivity': 'element_index',
})

# 2. The face-node connectivity array needs cf_role and start_index
ds_ugrid['element_index'].attrs.update({
    'cf_role': 'face_node_connectivity',
    'start_index': 1,        # SHYFEM uses 1-based node indices
})

# 3. Node coordinate variables need cf_role
ds_ugrid['longitude'].attrs['standard_name'] = 'longitude'
ds_ugrid['latitude'].attrs['standard_name'] = 'latitude'

# Promote lon/lat to coordinates so xugrid can find them
ds_ugrid = ds_ugrid.set_coords(['longitude', 'latitude', 'element_index', 'total_depth', 'topology'])

print(ds_ugrid)

In [ ]:
# Now open as a UgridDataset — xugrid detects the mesh_topology variable automatically
uds = xu.UgridDataset.from_dataset(ds_ugrid)
uds

### 2.4 Inspect the mesh topology

In [ ]:
import matplotlib.pyplot as plt

grid = uds.ugrid.grid
print(f"Mesh type:     {type(grid).__name__}")
print(f"Nodes:         {grid.n_node:,}")
print(f"Faces (tris):  {grid.n_face:,}")
print(f"Edges:         {grid.n_edge:,}")

fig, ax = plt.subplots(figsize=(7, 5))
grid.plot(ax=ax, linewidth=0.3)
ax.set_title('Antsiranana Bay — SHYFEM triangular mesh')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()

### 2.5 Plot on the native unstructured mesh

In [ ]:
# Water level at last time step — native triangular mesh, no regridding
wl_t = uds['water_level'].isel(time=-1).load()

wl_t.ugrid.plot(cmap='RdBu_r', figsize=(8, 6))
plt.title(f'Water Level (m) — {str(uds.time.values[-1])[:10]}')
plt.show()

In [ ]:
# Surface temperature — demonstrates xarray ops preserve the UGRID grid
temp_surf = uds['temperature'].isel(level=0, time=-1).load()

temp_surf.ugrid.plot(cmap='plasma', figsize=(8, 6))
plt.title('Surface Temperature (°C)')
plt.show()

### 2.6 xarray operations preserve the grid

Standard xarray reductions work on `UgridDataArray` — the UGRID topology is carried along automatically.

In [ ]:
# Time-mean surface temperature
temp_mean = uds['temperature'].isel(level=0).mean('time').load()

print(f"Result type: {type(temp_mean).__name__}")

temp_mean.ugrid.plot(cmap='plasma', figsize=(8, 6))
plt.title('Time-mean Surface Temperature (°C)')
plt.show()

### 2.7 Spatial clipping

In [ ]:
# Clip to inner bay — topology is automatically subsetted
uds_bay = uds.ugrid.clip_box(xmin=49.25, ymin=-12.40, xmax=49.40, ymax=-12.25)

print(f"Full mesh:    {uds.ugrid.grid.n_face:,} faces")
print(f"Clipped mesh: {uds_bay.ugrid.grid.n_face:,} faces")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
uds['water_level'].isel(time=-1).load().ugrid.plot(ax=axes[0], cmap='RdBu_r')
axes[0].set_title('Full domain')
uds_bay['water_level'].isel(time=-1).load().ugrid.plot(ax=axes[1], cmap='RdBu_r')
axes[1].set_title('Inner bay (clipped)')
plt.tight_layout()
plt.show()

### 2.8 Regridding to a structured grid

A common workflow: **native unstructured mesh → regular lat/lon grid** for post-processing or comparison with satellite data.

In [ ]:
# Build a regular target grid covering the domain
lon_range = np.linspace(float(ds.longitude.min()), float(ds.longitude.max()), 200)
lat_range = np.linspace(float(ds.latitude.min()), float(ds.latitude.max()), 200)
target_grid = xu.Ugrid2d.from_structured(lon_range, lat_range)

# Regrid using centroid locator
regridder = xu.CentroidLocatorRegridder(source=uds, target=target_grid)
temp_regridded = regridder.regrid(uds['temperature'].isel(level=0, time=-1).load())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
uds['temperature'].isel(level=0, time=-1).load().ugrid.plot(ax=axes[0], cmap='plasma')
axes[0].set_title('Native unstructured mesh')
temp_regridded.ugrid.plot(ax=axes[1], cmap='plasma')
axes[1].set_title('Regridded to 200×200 regular grid')
plt.tight_layout()
plt.show()

### 2.9 Export back to NetCDF with full UGRID attributes

In [ ]:
# Convert back to a plain xr.Dataset with all UGRID CF attributes written out
ds_out = uds.ugrid.to_dataset()

# Verify the UGRID attributes are now present
print("mesh_topology cf_role:          ", ds_out['topology'].attrs.get('cf_role'))
print("face_node_connectivity pointer: ", ds_out['topology'].attrs.get('face_node_connectivity'))
print("node_coordinates pointer:       ", ds_out['topology'].attrs.get('node_coordinates'))

---
## Summary

### CF Conventions + cf_xarray

| Task | Without CF | With cf_xarray |
|---|---|---|
| Get longitude | `ds['longitude']` (if you know the name) | `ds.cf['longitude']` |
| Get temperature | `ds['temperature']` | `ds.cf['sea_water_temperature']` |
| Select by time | `ds.sel(time='2021-04')` | `ds.cf.sel(T='2021-04')` |
| Select depth | `ds.sel(level=10)` | `ds.cf.sel(Z=10)` |
| Works across models | No — names vary | Yes — `standard_name` is universal |

### UGRID Conventions + xugrid

| Task | Without xugrid | With xugrid |
|---|---|---|
| Open UGRID file | `xr.open_dataset()` (raw arrays) | `xu.open_dataset()` (topology-aware) |
| Plot on native mesh | manual triangulation + datashader | `.ugrid.plot()` |
| Spatial clip | manual index math | `.ugrid.clip_box()` |
| Regrid to structured | scipy + manual | `xu.CentroidLocatorRegridder` |
| xarray time ops | lose grid context | grid preserved automatically |

### Resources

- [CF Conventions](https://cfconventions.org/) and [Standard Names table](https://cfconventions.org/standard-names.html)
- [cf_xarray docs](https://cf-xarray.readthedocs.io/)
- [UGRID Conventions](https://ugrid-conventions.github.io/ugrid-conventions/)
- [xugrid docs](https://deltares.github.io/xugrid/) and [examples gallery](https://deltares.github.io/xugrid/examples/)